In [1]:
!pip -q install nilearn pandas numpy matplotlib seaborn networkx
import numpy as np
import pandas as pd
import networkx as nx
from nilearn import datasets
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 55.3 MB/s eta 0:00:00


In [2]:
adhd = datasets.fetch_adhd(n_subjects=5)
func_files = adhd.func
print("Subjects loaded:", len(func_files))

[fetch_adhd] Added README.md to /root/nilearn_data

[fetch_adhd] Dataset created in /root/nilearn_data/adhd

[fetch_adhd] Downloading data from https://www.nitrc.org/frs/download.php/7781/adhd40_metadata.tgz ...

[fetch_adhd]  ...done. (0 seconds, 0 min)

[fetch_adhd] Extracting data from /root/nilearn_data/adhd/fbef5baff0b388a8c913a08e1d84e059/adhd40_metadata.tgz...

...

[fetch_adhd] Downloading data from https://www.nitrc.org/frs/download.php/7786/adhd40_0023008.tgz ...

[fetch_adhd]  ...done. (0 seconds, 0 min)

[fetch_adhd] Extracting data from /root/nilearn_data/adhd/9dc0563e798ef2ebbc2141b4fac6286a/adhd40_0023008.tgz...

[fetch_adhd] .. done.

Subjects loaded: 5


In [3]:
atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100,
    yeo_networks=7
)

masker = NiftiLabelsMasker(
    labels_img=atlas.maps,
    standardize="zscore_sample",
    verbose=0
)

corr = ConnectivityMeasure(
    kind="correlation",
    standardize="zscore_sample"
)

[fetch_atlas_schaefer_2018] Dataset created in /root/nilearn_data/schaefer_2018

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order.txt ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

[fetch_atlas_schaefer_2018] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_100Parcels_7Networks_order_FSLMNI152_1mm.
nii.gz ...

[fetch_atlas_schaefer_2018]  ...done. (0 seconds, 0 min)

In [4]:
mats = []

for f in func_files:
    ts = masker.fit_transform(f)
    mat = corr.fit_transform([ts])[0]
    mats.append(mat)

print("Matrices created:", len(mats))
print("Matrix shape:", mats[0].shape)

Matrices created: 5
Matrix shape: (100, 100)


In [5]:
def graph_metrics(mat, threshold=0.2):
    adj = (mat > threshold).astype(int)
    np.fill_diagonal(adj, 0)

    G = nx.from_numpy_array(adj)

    return {
        "Nodes": G.number_of_nodes(),
        "Edges": G.number_of_edges(),
        "Density": nx.density(G),
        "Clustering": nx.average_clustering(G),
        "Components": nx.number_connected_components(G)
    }

In [6]:
metrics = []

for i, mat in enumerate(mats):
    vals = graph_metrics(mat)
    vals["Subject"] = f"sub-{i+1}"
    metrics.append(vals)

metrics_df = pd.DataFrame(metrics)
metrics_df

,Nodes,Edges,Density,Clustering,Components,Subject
0,100,4650,0.939394,0.975703,1,sub-1
1,100,4849,0.979596,0.989596,2,sub-2
2,100,4318,0.872323,0.941182,1,sub-3
3,100,4727,0.954949,0.983686,1,sub-4
4,100,4316,0.871919,0.927053,1,sub-5


In [7]:
print("Notebook 2 complete.")
print("Connectivity matrices and topology metrics generated.")

Notebook 2 complete.
Connectivity matrices and topology metrics generated.
